In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Test") \
    .config("spark.driver.extraJavaOptions", "--add-opens=java.base/java.lang=ALL-UNNAMED") \
    .config("spark.executor.extraJavaOptions", "--add-opens=java.base/java.lang=ALL-UNNAMED") \
    .getOrCreate()

print("Spark started")
#Visualization has been done via PowerBI, hence there is no visualization code here

Spark started


In [4]:
files = {
    "full_grouped": "full_grouped.csv",
    "covid_clean": "covid_19_clean_complete.csv",
    "country_latest": "country_wise_latest.csv",
    "day_wise": "day_wise.csv",
    "usa_county": "usa_county_wise.csv",
    "worldometer": "worldometer_data.csv"
}

In [5]:
dfs = {}
for name, path in files.items():
    df = spark.read.csv(
        path,
        header=True,
        inferSchema=True
    )
    dfs[name] = df

for name, df in dfs.items():
    print(f" {name.upper()} ")
    print("Schema: ")

    df.printSchema()

    print("Row Count: ")
    print(df.count())



 FULL_GROUPED 
Schema: 
root
 |-- Date: date (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- New cases: integer (nullable = true)
 |-- New deaths: integer (nullable = true)
 |-- New recovered: integer (nullable = true)
 |-- WHO Region: string (nullable = true)

Row Count: 
35156
 COVID_CLEAN 
Schema: 
root
 |-- Province/State: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Long: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- WHO Region: string (nullable = true)

Row Count: 
49068
 COUNTRY_LATEST 
Schema: 
root
 |-- Country/Region: string (nullable = true)
 |-- Confirm

In [6]:
#2.3
from pyspark.sql.functions import col, when, regexp_replace
full_grouped = dfs['full_grouped']
country_wise_latest = dfs['country_latest']
worldometer_data = dfs['worldometer']

def standardize_country_names(df, column_name):
    df = df.withColumn(
        column_name,
        when(col(column_name) == "US", "USA")
        .when(col(column_name) == "Korea, South", "South Korea")
        .when(col(column_name) == "UK", "United Kingdom")
        .otherwise(col(column_name)) 

    )

    df = df.withColumn(
        column_name,
        regexp_replace(col(column_name), "Mainland China", "China")

    )

    return df

print(type(full_grouped))
full_grouped_cleaned = standardize_country_names(
    full_grouped,
    "Country/Region"
)

country_wise_latest_cleaned = standardize_country_names(
    country_wise_latest,
    "Country/Region"
)

worldometer_data_cleaned = standardize_country_names(
    worldometer_data,
    "Country/Region"
)

print("Full Grouped Countries: ")
full_grouped_cleaned.select("Country/Region").distinct().show()
print("Country Wise Latest Countries:")
country_wise_latest_cleaned.select("Country/Region").distinct().show()

print("Worldometer Countries:")
worldometer_data_cleaned.select("Country/Region").distinct().show()



<class 'pyspark.sql.classic.dataframe.DataFrame'>
Full Grouped Countries: 
+--------------+
|Country/Region|
+--------------+
|          Chad|
|      Paraguay|
|        Russia|
|         Yemen|
|       Senegal|
|    Cabo Verde|
|        Sweden|
|        Guyana|
|         Burma|
|       Eritrea|
|   Philippines|
|      Djibouti|
|      Malaysia|
|     Singapore|
|          Fiji|
|        Turkey|
|        Malawi|
|Western Sahara|
|          Iraq|
|       Germany|
+--------------+
only showing top 20 rows
Country Wise Latest Countries:
+--------------+
|Country/Region|
+--------------+
|          Chad|
|      Paraguay|
|        Russia|
|         Yemen|
|       Senegal|
|    Cabo Verde|
|        Sweden|
|        Guyana|
|         Burma|
|       Eritrea|
|   Philippines|
|      Djibouti|
|      Malaysia|
|     Singapore|
|          Fiji|
|        Turkey|
|        Malawi|
|Western Sahara|
|          Iraq|
|       Germany|
+--------------+
only showing top 20 rows
Worldometer Countries:
+----

In [ ]:


import re


# Function to standardize column names


def standardize_column_names(df):

    for old_col in df.columns:

        # lowercase
        new_col = old_col.lower()

        # replace spaces and special chars
        new_col = re.sub(r"[ /.-]+", "_", new_col)

        # remove brackets and commas
        new_col = re.sub(r"[(),]", "", new_col)

        # remove duplicate underscores
        new_col = re.sub(r"_+", "_", new_col)

        # remove leading/trailing underscores
        new_col = new_col.strip("_")

        # rename
        df = df.withColumnRenamed(old_col, new_col)

    return df




standardized_dfs = {}

for name, df in dfs.items():

    standardized_df = standardize_column_names(df)

    standardized_dfs[name] = standardized_df

    print(f"\n{name.upper()} STANDARDIZED")
    print(standardized_df.columns)


FULL_GROUPED STANDARDIZED
['date', 'country_region', 'confirmed', 'deaths', 'recovered', 'active', 'new_cases', 'new_deaths', 'new_recovered', 'who_region']

COVID_CLEAN STANDARDIZED
['province_state', 'country_region', 'lat', 'long', 'date', 'confirmed', 'deaths', 'recovered', 'active', 'who_region']

COUNTRY_LATEST STANDARDIZED
['country_region', 'confirmed', 'deaths', 'recovered', 'active', 'new_cases', 'new_deaths', 'new_recovered', 'deaths_100_cases', 'recovered_100_cases', 'deaths_100_recovered', 'confirmed_last_week', '1_week_change', '1_week_%_increase', 'who_region']

DAY_WISE STANDARDIZED
['date', 'confirmed', 'deaths', 'recovered', 'active', 'new_cases', 'new_deaths', 'new_recovered', 'deaths_100_cases', 'recovered_100_cases', 'deaths_100_recovered', 'no_of_countries']

USA_COUNTY STANDARDIZED
['uid', 'iso2', 'iso3', 'code3', 'fips', 'admin2', 'province_state', 'country_region', 'lat', 'long', 'combined_key', 'date', 'confirmed', 'deaths']

WORLDOMETER STANDARDIZED
['countr

In [ ]:


import re


def standardize_column_names(df):

    for old_col in df.columns:

   
        new_col = old_col.lower()

     
        new_col = re.sub(r"[ /.-]+", "_", new_col)

       
        new_col = re.sub(r"[(),]", "", new_col)

        
        new_col = re.sub(r"_+", "_", new_col)

        
        new_col = new_col.strip("_")

        df = df.withColumnRenamed(old_col, new_col)

    return df



standardized_dfs = {}

for name, df in dfs.items():

    standardized_df = standardize_column_names(df)

    standardized_dfs[name] = standardized_df

    print(f"\n{name.upper()} STANDARDIZED")
    print(standardized_df.columns)


FULL_GROUPED STANDARDIZED
['date', 'country_region', 'confirmed', 'deaths', 'recovered', 'active', 'new_cases', 'new_deaths', 'new_recovered', 'who_region']

COVID_CLEAN STANDARDIZED
['province_state', 'country_region', 'lat', 'long', 'date', 'confirmed', 'deaths', 'recovered', 'active', 'who_region']

COUNTRY_LATEST STANDARDIZED
['country_region', 'confirmed', 'deaths', 'recovered', 'active', 'new_cases', 'new_deaths', 'new_recovered', 'deaths_100_cases', 'recovered_100_cases', 'deaths_100_recovered', 'confirmed_last_week', '1_week_change', '1_week_%_increase', 'who_region']

DAY_WISE STANDARDIZED
['date', 'confirmed', 'deaths', 'recovered', 'active', 'new_cases', 'new_deaths', 'new_recovered', 'deaths_100_cases', 'recovered_100_cases', 'deaths_100_recovered', 'no_of_countries']

USA_COUNTY STANDARDIZED
['uid', 'iso2', 'iso3', 'code3', 'fips', 'admin2', 'province_state', 'country_region', 'lat', 'long', 'combined_key', 'date', 'confirmed', 'deaths']

WORLDOMETER STANDARDIZED
['countr

In [ ]:


from pyspark.sql.functions import (
    col,
    trim,
    when,
    count,
    isnan
)

from pyspark.sql.types import (
    IntegerType,
    DoubleType,
    FloatType,
    LongType,
    ShortType,
    DecimalType
)



def clean_missing_values(df):

    
    df = df.dropDuplicates()

 
    for column_name, dtype in df.dtypes:

        if dtype == "string":

            df = df.withColumn(
                column_name,
                trim(col(column_name))
            )


    numeric_types = (
        IntegerType,
        DoubleType,
        FloatType,
        LongType,
        ShortType,
        DecimalType
    )

    numeric_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, numeric_types)
    ]


    if numeric_columns:

        df = df.fillna(
            0,
            subset=numeric_columns
        )


    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    # Fill string nulls
    if string_columns:

        df = df.fillna(
            "Unknown",
            subset=string_columns
        )

    return df



# Apply cleaning

cleaned_dfs = {}

for name, df in standardized_dfs.items():

    cleaned_df = clean_missing_values(df)

    cleaned_dfs[name] = cleaned_df

    print(f"\n{name.upper()} CLEANED")

    expressions = []

    for field in cleaned_df.schema.fields:

        column_name = field.name

        # Numeric columns
        if isinstance(field.dataType, (
            IntegerType,
            DoubleType,
            FloatType,
            LongType,
            ShortType,
            DecimalType
        )):

            expr = count(
                when(
                    col(column_name).isNull() |
                    isnan(col(column_name)),
                    column_name
                )
            ).alias(column_name)

        
        else:

            expr = count(
                when(
                    col(column_name).isNull(),
                    column_name
                )
            ).alias(column_name)

        expressions.append(expr)

    cleaned_df.select(expressions).show()


FULL_GROUPED CLEANED
+----+--------------+---------+------+---------+------+---------+----------+-------------+----------+
|date|country_region|confirmed|deaths|recovered|active|new_cases|new_deaths|new_recovered|who_region|
+----+--------------+---------+------+---------+------+---------+----------+-------------+----------+
|   0|             0|        0|     0|        0|     0|        0|         0|            0|         0|
+----+--------------+---------+------+---------+------+---------+----------+-------------+----------+


COVID_CLEAN CLEANED
+--------------+--------------+---+----+----+---------+------+---------+------+----------+
|province_state|country_region|lat|long|date|confirmed|deaths|recovered|active|who_region|
+--------------+--------------+---+----+----+---------+------+---------+------+----------+
|             0|             0|  0|   0|   0|        0|     0|        0|     0|         0|
+--------------+--------------+---+----+----+---------+------+---------+------+---

In [10]:
dfs = cleaned_dfs

In [11]:
import matplotlib.pyplot as plt

country_wise = dfs["country_latest"]
top_10 = country_wise.select(
    "Country_Region",
    "Confirmed"
).orderBy(
    col("Confirmed").desc()
).limit(10)

top_10.show()



# Save as CSV
top_10.toPandas().to_csv(
    "outputs/top10_confirmed.csv",
    index=False
)


+--------------+---------+
|Country_Region|Confirmed|
+--------------+---------+
|            US|  4290259|
|        Brazil|  2442375|
|         India|  1480073|
|        Russia|   816680|
|  South Africa|   452529|
|        Mexico|   395489|
|          Peru|   389717|
|         Chile|   347923|
|United Kingdom|   301708|
|          Iran|   293606|
+--------------+---------+



In [12]:
country_wise = country_wise.withColumn("death_rate", (col("Deaths")) / (col("Confirmed")) * 100)
top_10_death_rate = country_wise.select(
    "Country_Region",
    "death_rate"

).orderBy(
    col("death_rate").desc()
).limit(10)

top_10_death_rate.toPandas().to_csv(
    "outputs/top10_death_rate.csv",
    index=False
)


In [13]:
from pyspark.sql.functions import sum as spark_sum
full_grouped = dfs["full_grouped"]


who_region_totals = full_grouped.groupBy("who_region").agg(
    spark_sum("Confirmed").alias("Total_Confirmed"),
    spark_sum("Deaths").alias("Total_Deaths"),
    spark_sum("Recovered").alias("Total_Recovered")

)


who_region_totals.show()

who_region_totals.toPandas().to_csv(
    "outputs/who_region_totals.csv",
    index=False
)


+--------------------+---------------+------------+---------------+
|          who_region|Total_Confirmed|Total_Deaths|Total_Recovered|
+--------------------+---------------+------------+---------------+
|              Europe|      248879793|    19271040|      123202075|
|     Western Pacific|       26374411|      932430|       18861950|
|              Africa|       21791827|      439978|       11193730|
|Eastern Mediterra...|       74082892|     1924029|       48050703|
|            Americas|      402261194|    19359292|      157069444|
|     South-East Asia|       55118365|     1458134|       30030327|
+--------------------+---------------+------------+---------------+



In [14]:
day_wise = dfs["day_wise"]

daily_cases = day_wise.select("date", "new_cases").orderBy("date")
daily_cases.show()
daily_cases.toPandas().to_csv(
    "outputs/daily_cases.csv",
    index=False
)


+----------+---------+
|      date|new_cases|
+----------+---------+
|2020-01-22|        0|
|2020-01-23|       99|
|2020-01-24|      287|
|2020-01-25|      493|
|2020-01-26|      684|
|2020-01-27|      809|
|2020-01-28|     2651|
|2020-01-29|      588|
|2020-01-30|     2068|
|2020-01-31|     1693|
|2020-02-01|     2111|
|2020-02-02|     4749|
|2020-02-03|     3100|
|2020-02-04|     4011|
|2020-02-05|     3745|
|2020-02-06|     3159|
|2020-02-07|     3532|
|2020-02-08|     2734|
|2020-02-09|     3027|
|2020-02-10|     2538|
+----------+---------+
only showing top 20 rows


In [15]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, when



window_spec = Window.orderBy("Date")
day_wise_growth = day_wise.withColumn("previous_day_deaths", lag("Deaths").over(window_spec))

day_wise_growth = day_wise_growth.withColumn(
    "death_growth_percentage",
    when(
        col("previous_day_deaths") != 0,
        (col("new_deaths") / col("previous_day_deaths")) * 100

    ).otherwise(0)
)

growth_trend = day_wise_growth.select(
    "date",
    "new_deaths",
    "previous_day_deaths",
    "death_growth_percentage"
)

growth_trend.show()

growth_trend.toPandas().to_csv(
    "outputs/growth_trend.csv",
    index=False
)






+----------+----------+-------------------+-----------------------+
|      date|new_deaths|previous_day_deaths|death_growth_percentage|
+----------+----------+-------------------+-----------------------+
|2020-01-22|         0|               NULL|                    0.0|
|2020-01-23|         1|                 17|       5.88235294117647|
|2020-01-24|         8|                 18|      44.44444444444444|
|2020-01-25|        16|                 26|      61.53846153846154|
|2020-01-26|        14|                 42|      33.33333333333333|
|2020-01-27|        26|                 56|      46.42857142857143|
|2020-01-28|        49|                 82|     59.756097560975604|
|2020-01-29|         2|                131|     1.5267175572519083|
|2020-01-30|        38|                133|      28.57142857142857|
|2020-01-31|        42|                171|     24.561403508771928|
|2020-02-01|        46|                213|       21.5962441314554|
|2020-02-02|       103|                259|     

In [16]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank
import matplotlib.pyplot as plt
import pandas as pd

window_spec = Window.partitionBy("who_region").orderBy(col("confirmed").desc())
ranked = country_wise.withColumn(
    "rank",
    dense_rank().over(window_spec)
)
top_5_per_region = ranked.filter(
    col("rank") <= 5
).select(
    "who_region",
    "country_region",
    "confirmed",
    "rank"
).orderBy(
    "who_region",
    "rank"
)

top_5_per_region.show()

top_5_per_region.toPandas().to_csv(
    "outputs/top_5_per_region.csv",
    index=False
)



+--------------------+--------------+---------+----+
|          who_region|country_region|confirmed|rank|
+--------------------+--------------+---------+----+
|              Africa|  South Africa|   452529|   1|
|              Africa|       Nigeria|    41180|   2|
|              Africa|         Ghana|    33624|   3|
|              Africa|       Algeria|    27973|   4|
|              Africa|         Kenya|    17975|   5|
|            Americas|            US|  4290259|   1|
|            Americas|        Brazil|  2442375|   2|
|            Americas|        Mexico|   395489|   3|
|            Americas|          Peru|   389717|   4|
|            Americas|         Chile|   347923|   5|
|Eastern Mediterra...|          Iran|   293606|   1|
|Eastern Mediterra...|      Pakistan|   274289|   2|
|Eastern Mediterra...|  Saudi Arabia|   268934|   3|
|Eastern Mediterra...|          Iraq|   112585|   4|
|Eastern Mediterra...|         Qatar|   109597|   5|
|              Europe|        Russia|   816680

In [17]:
window_spec = Window.partitionBy(
    "country_region"
).orderBy("Date")

daily_increase_df = full_grouped.withColumn(
    "yesterday_confirmed",
    lag("confirmed").over(window_spec)
)

daily_increase_df = daily_increase_df.withColumn(
    "daily_case_increase",
    col("confirmed") - col("yesterday_confirmed")

)
daily_increase_df = daily_increase_df.select(
    "country_region",
    "date",
    "confirmed",
    "yesterday_confirmed",
    "daily_case_increase"
)


daily_increase_df.show()

daily_increase_df.toPandas().to_csv(
    "outputs/daily_increase.csv",
    index=False
)





+--------------+----------+---------+-------------------+-------------------+
|country_region|      date|confirmed|yesterday_confirmed|daily_case_increase|
+--------------+----------+---------+-------------------+-------------------+
|   Afghanistan|2020-01-22|        0|               NULL|               NULL|
|   Afghanistan|2020-01-23|        0|                  0|                  0|
|   Afghanistan|2020-01-24|        0|                  0|                  0|
|   Afghanistan|2020-01-25|        0|                  0|                  0|
|   Afghanistan|2020-01-26|        0|                  0|                  0|
|   Afghanistan|2020-01-27|        0|                  0|                  0|
|   Afghanistan|2020-01-28|        0|                  0|                  0|
|   Afghanistan|2020-01-29|        0|                  0|                  0|
|   Afghanistan|2020-01-30|        0|                  0|                  0|
|   Afghanistan|2020-01-31|        0|                  0|       

In [24]:
from pyspark.sql.functions import col, when

worldometer_data = dfs["worldometer"]

worldometer_data = worldometer_data.withColumn(
    "infection_rate",
    when(
        col("population") != 0,
        (col("totalcases") / col("population")) * 100
    ).otherwise(0)
)

infection_rate_df = worldometer_data.select(
    "country_region",
    "infection_rate"
).orderBy(
    col("infection_rate").desc()
)

infection_rate_df.show()

top_10_infection = infection_rate_df.limit(10)

top_10_infection.show()

top_10_infection.toPandas().to_csv(
    "outputs/top_10_infection.csv",
    index=False
)


+--------------+------------------+
|country_region|    infection_rate|
+--------------+------------------+
|         Qatar|3.9921575750452756|
| French Guiana|2.7145648579588157|
|       Bahrain|2.5130239079751258|
|    San Marino|2.0596381637102956|
|         Chile|1.9164810228284688|
|        Panama| 1.652703989232825|
|        Kuwait|1.6378443167538763|
|          Oman|1.5769043963734304|
|           USA|1.5193862960518527|
|  Vatican City|1.4981273408239701|
|          Peru|1.3793451656436928|
|        Brazil|1.3716104125127853|
|       Armenia| 1.343506721582449|
|       Andorra| 1.221563705064831|
|    Luxembourg|1.1281565414896195|
|       Mayotte| 1.112578131000406|
|     Singapore|0.9317785415782796|
|  South Africa|0.9063149328193871|
|        Israel|0.8649983310845557|
|      Maldives|0.8643489310146126|
+--------------+------------------+
only showing top 20 rows
+--------------+------------------+
|country_region|    infection_rate|
+--------------+------------------+
|  

In [27]:
from pyspark.sql.functions import col, when
import matplotlib.pyplot as plt



country_wise = country_wise.withColumn(
    "recovery_rate",
    when(
        col("confirmed") != 0,
        (col("recovered") / col("confirmed")) * 100
    ).otherwise(0)
)

best_recovery = country_wise.select(
    "country_region",
    "recovery_rate"
).orderBy(
    col("recovery_rate").desc()
).limit(10)

worst_recovery = country_wise.select(
    "country_region",
    "recovery_rate"
).orderBy(
    col("recovery_rate").asc()
).limit(10)

best_recovery.show()

worst_recovery.show()

best_recovery.toPandas().to_csv(
    "outputs/best_recovery.csv",
    index=False
)
worst_recovery.toPandas().to_csv(
    "outputs/worst_recovery.csv",
    index=False
)



+--------------+-----------------+
|country_region|    recovery_rate|
+--------------+-----------------+
|       Grenada|            100.0|
|      Holy See|            100.0|
|      Dominica|            100.0|
|      Djibouti|98.37912630954733|
|       Iceland| 98.3279395900755|
|        Brunei|97.87234042553192|
|   New Zealand|97.23827874116891|
|         Qatar| 97.0172541219194|
|      Malaysia|96.59703504043127|
|     Mauritius|96.51162790697676|
+--------------+-----------------+

+--------------+-------------------+
|country_region|      recovery_rate|
+--------------+-------------------+
|        Serbia|                0.0|
|    Mozambique|                0.0|
|         Syria|                0.0|
|   Timor-Leste|                0.0|
|        Sweden|                0.0|
|        Canada|                0.0|
|   Netherlands|0.35384644187744557|
|United Kingdom|0.47628833176448754|
|       Namibia|  5.480195333695062|
|      Botswana|  8.525033829499323|
+--------------+------------